In [1]:
import sys
print("python:", sys.version.split()[0])
print("kernel:", sys.executable, "\n")

for name in ["coffea", "uproot", "awkward", "numpy", "matplotlib", "hist", "mplhep", "vector"]:
    try:
        mod = __import__(name)
        print(f"  [ok]      {name:<12s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  [MISSING] {name}")

python: 3.10.14
kernel: /depot/cms/conda_envs/bhanda25/Coffea-Santosh/bin/python 

  [ok]      coffea       2024.2.2
  [ok]      uproot       5.3.10
  [ok]      awkward      2.6.5
  [ok]      numpy        1.26.4
  [ok]      matplotlib   3.9.1
  [ok]      hist         2.7.3
  [ok]      mplhep       0.3.50
  [ok]      vector       1.4.1


In [2]:
import os
import pathlib
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

# coffea 2024.2.2 warns on import that a future release will hand its four-vector
# maths to `vector`, the package you used in Lab 1 (more on that in §1.3). Nothing
# we do is affected, so we silence that one warning. Silencing warnings you have
# not read is a bad habit; this one has been read.
warnings.filterwarnings("ignore", category=FutureWarning, module="coffea.*")

import awkward as ak
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import coffea
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, BaseSchema
from coffea.nanoevents.methods import candidate

from utils import chunk_jobs, plot_dimuon_spectrum

hep.style.use("CMS")
print("coffea", coffea.__version__)

coffea 2024.2.2


In [3]:
import json
import urllib.request

RECID = 75589
with urllib.request.urlopen(f"https://opendata.cern.ch/api/records/{RECID}", timeout=60) as r:
    meta = json.load(r)["metadata"]

print("title      :", meta["title"])
print("DOI        :", meta["doi"])
print("licence    :", meta["license"])
print("distribution:", json.dumps(meta["distribution"], indent=2))
meta_files = []
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        #print("Files:", meta["_file_indices"][i]['files'][j])
        meta_files.append(meta["_file_indices"][i]['files'][j])
print(meta_files[0:3])

title      : /ZZTo4L_TuneCP5_13TeV_powheg_pythia8/RunIISummer20UL16NanoAODv9-106X_mcRun2_asymptotic_v17-v1/NANOAODSIM
DOI        : 10.7483/OPENDATA.CMS.3Z2W.O02W
licence    : {'attribution': 'CC0-1.0'}
distribution: {
  "formats": [
    "nanoaodsim",
    "root"
  ],
  "number_events": 52104000,
  "number_files": 99,
  "size": 71479443458
}
[{'availability': 'online', 'bucket': 'db3870f1-c425-4e5e-83a3-6350604fa413', 'checksum': 'adler32:6af0d28f', 'file_id': '227c3166-3a84-4c93-b561-9e60463cc0c4', 'filename': '051E9D22-4F30-8E49-8477-644E65768282.root', 'key': 'CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v1_2430000_file_index.json_0', 'size': 232295116, 'tags': {}, 'uri': 'root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/ZZTo4L_TuneCP5_13TeV_powheg_pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/2430000/051E9D22-4F30-8E49-8477-644E65768282.root', 'version_id': '3296448b-1ed8-401b-884c-f0256bdc4ea7'

In [14]:
# !!! ONLY RUN THIS 1 TIME EVER UNLESS YOU WANT TO WAIT 1 HOUR TO DO ANYTHING !!! #
import os, pathlib, shutil, subprocess, time
from XRootD import client
import urllib.request

for i in range(len(meta_files)):
    XROOTD_URL = meta_files[i]["uri"]
    HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
    EXPECTED   = meta_files[i]["size"]
    
    DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_background_data"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE = DATA_DIR / meta_files[i]["key"]
    
    if LOCAL_FILE.exists() and LOCAL_FILE.stat().st_size == EXPECTED:
        print("Already downloaded:", LOCAL_FILE)
    else:
        t0, ok = time.time(), False
    
        if shutil.which("xrdcp"):
            print("Trying xrdcp ...")
            ok = subprocess.run(["xrdcp", "-f", XROOTD_URL, str(LOCAL_FILE)]).returncode == 0
    
        if not ok:
            try:
                print("Trying XRootD python bindings ...")
                ok = client.FileSystem("root://eospublic.cern.ch").copy(
                         XROOTD_URL, str(LOCAL_FILE), force=True)[0].ok
            except ImportError:
                pass
    
        if not ok:
            print("Falling back to HTTPS ...")
            urllib.request.urlretrieve(HTTPS_URL, LOCAL_FILE)
    
        print(f"done in {time.time()-t0:.0f} s -> {LOCAL_FILE}")
    
    print(f"\nsize on disk: {LOCAL_FILE.stat().st_size:,} bytes")
    print("Loops:", i)

Trying XRootD python bindings ...
done in 17 s -> /scratch/scholar/evans793/higgs_background_data/CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v1_2430000_file_index.json_0

size on disk: 232,295,116 bytes
Loops: 0
Trying XRootD python bindings ...
done in 5 s -> /scratch/scholar/evans793/higgs_background_data/CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v1_2430000_file_index.json_1

size on disk: 117,682,395 bytes
Loops: 1
Trying XRootD python bindings ...
done in 56 s -> /scratch/scholar/evans793/higgs_background_data/CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v1_2430000_file_index.json_2

size on disk: 1,379,926,833 bytes
Loops: 2
Trying XRootD python bindings ...
done in 57 s -> /scratch/scholar/evans793/higgs_background_data/CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_

In [6]:
import zlib
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        # YOUR CODE HERE
        XROOTD_URL = meta_files[i]["uri"]
        HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
        EXPECTED   = meta_files[i]["size"]
        
        DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_background_data"
        # DATA_DIR.mkdir(parents=True, exist_ok=True)
        LOCAL_FILE = DATA_DIR / meta["_file_indices"][i]["files"][j]["key"]
                  
        local_size = LOCAL_FILE.stat().st_size
        print (f"Filesize (bytes): {local_size:,}")
        print ("Filesize (MB): " + str(round(local_size/1000000, 2)))
        print("Size matches:", local_size == meta["_file_indices"][i]["files"][j]["size"])
        
        running_value = 1
        with open(LOCAL_FILE, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                running_value = zlib.adler32(chunk, running_value)
        checksum = f"{running_value:08x}"
        expected_checksum = meta["_file_indices"][i]["files"][j]["checksum"].replace("adler32:", "")
        print("Checksum matches: " + str(checksum == expected_checksum))

Filesize (bytes): 232,295,116
Filesize (MB): 232.3
Size matches: True
Checksum matches: True
Filesize (bytes): 117,682,395
Filesize (MB): 117.68
Size matches: True
Checksum matches: True
Filesize (bytes): 1,379,926,833
Filesize (MB): 1379.93
Size matches: True
Checksum matches: True
Filesize (bytes): 1,892,690,275
Filesize (MB): 1892.69
Size matches: True
Checksum matches: True
Filesize (bytes): 117,777,043
Filesize (MB): 117.78
Size matches: True
Checksum matches: True
Filesize (bytes): 1,380,173,008
Filesize (MB): 1380.17
Size matches: True
Checksum matches: True
Filesize (bytes): 232,389,277
Filesize (MB): 232.39
Size matches: True
Checksum matches: True
Filesize (bytes): 19,424,842
Filesize (MB): 19.42
Size matches: True
Checksum matches: True
Filesize (bytes): 15,093,676
Filesize (MB): 15.09
Size matches: True
Checksum matches: True
Filesize (bytes): 1,264,692,502
Filesize (MB): 1264.69
Size matches: True
Checksum matches: True
Filesize (bytes): 7,030,316
Filesize (MB): 7.03
Size 

In [ ]:
# /eos/opendata/cms/dataset-semantics/NanoAODSIM/75589/ZZTo4L_TuneCP5_13TeV_powheg_pythia8_doc.root